In [1]:
lapply(c('viridis', 'ggthemes', 'skimr'),
       function(pkg_name) { if(! pkg_name %in% installed.packages()) { install.packages(pkg_name)} } )

library(viridis)    # A nice color scheme for plots.
library(ggthemes)   # Common themes to change the look and feel of plots.
library(scales)     # Graphical scales map data to aesthetics in plots.
library(skimr)      # Better summaries of data.
library(lubridate)  # Date library from the tidyverse.
library(tidyverse)  # Data wrangling packages.
library(bigrquery)  # Data extraction from Google BigQuery
library(data.table )

# Load required libraries
library(dplyr)
library(tidyr)
library(mice)


## Plot setup.
theme_set(theme_bw(base_size = 14)) # Default theme for plots.

#' Returns a data frame with a y position and a label, for use annotating ggplot boxplots.
#'
#' @param d A data frame.
#' @return A data frame with column y as max and column label as length.
get_boxplot_fun_data <- function(df) {
  return(data.frame(y = max(df), label = stringr::str_c('N = ', length(df))))
}

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

[[1]]
NULL

[[2]]
NULL

[[3]]
NULL

Loading required package: viridisLite


Attaching package: ‘scales’


The following object is masked from ‘package:viridis’:

    viridis_pal



Attaching package: ‘lubridate’


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr   1.1.4     ✔ readr   2.1.5
✔ forcats 1.0.0     ✔ stringr 1.5.1
✔ ggplot2 3.5.2     ✔ tibble  3.2.1
✔ purrr   1.0.4     ✔ tidyr   1.3.1
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ readr::col_factor() masks scales::col_factor()
✖ purrr::discard()    masks scales::discard()
✖ dplyr::filter()     masks stats::filter()
✖ dplyr::lag()        masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last


The following object is

In [2]:
name_of_file_in_bucket <- 'All_SDoH_data_domain_filtered_60.csv'

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ", my_bucket, "/data/", name_of_file_in_bucket, " ."), intern=T)

# Load the file into a dataframe
sdoh_data  <- read_csv(name_of_file_in_bucket)

character(0)

Rows: 54313 Columns: 54
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr   (6): SexGender, where_born, military, healthcare, disabled, sexual_ori...
dbl  (37): person_id, race_unknown, age_today, LGBTQIA, ehr_length, relative...
lgl   (8): AIAN, Asian, Black, Mid, Multiple, PI, White, His
date  (3): date_of_birth, min_date, max_date

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [3]:
#Combine small group
sdoh_data <- sdoh_data %>%
  mutate(AANHPI = (PI == 1 | Asian == 1))

exclude<-c("date_of_birth", "age_today", 'where_born',
           'military', 'healthcare', 'disabled', 'sexual_orientation', 'LGBTQIA',
           'min_date', 'max_date', 'ehr_length', 'relative_health', "PI", "AIAN", "Asian",
          'employment', 'income', 'number_living')
sdoh_data <- sdoh_data |> select(-exclude)

Warning message:
“Using an external vector in selections was deprecated in tidyselect 1.1.0.
ℹ Please use `all_of()` or `any_of()` instead.
  # Was:
  data %>% select(exclude)

  # Now:
  data %>% select(all_of(exclude))

See <https://tidyselect.r-lib.org/reference/faq-external-vector.html>.”


In [4]:
names(sdoh_data)

[1] "person_id"                 "Black"                    
 [3] "Mid"                       "Multiple"                 
 [5] "White"                     "His"                      
 [7] "race_unknown"              "SexGender"                
 [9] "record_depth"              "visit_frequency"          
[11] "age"                       "Education"                
[13] "Own Home"                  "Health Literacy"          
[15] "Housing Quality"           "Housing Instability"      
[17] "Food Insecurity"           "Walkability"              
[19] "Loneliness"                "Crime"                    
[21] "Physical Disorder"         "Social Disorder"          
[23] "Everyday Discrimination"   "Medical Discrimination"   
[25] "Social Cohesion"           "Stress"                   
[27] "Social Support"            "Spiritual Experiences"    
[29] "Health Coverage"           "Healthcare Utilization"   
[31] "Delayed Care"              "Can't afford care"        
[33] "Worried Pay"               "Respect"                  
[35] "Percent Poverty Threshold" "age2"                     
[37] "AANHPI"

In [9]:
factors<-c('AANHPI', 'Black', 'Mid', 'Multiple', 'White', 'His', 'race_unknown', 'SexGender')

sdoh_data<- sdoh_data %>%
  mutate(across(all_of(factors), as.factor))

In [10]:
missing_table <- enframe(sapply(sdoh_data, function(x) 
  round(sum(is.na(x)) / nrow(sdoh_data) * 100, 2)),
  name = "variable",
  value = "percent_missing"
) %>%
  arrange(percent_missing)

missing_table


variable,percent_missing
<chr>,<dbl>
person_id,0.00
Black,0.00
Mid,0.00
Multiple,0.00
White,0.00
His,0.00
race_unknown,0.00
SexGender,0.00
record_depth,0.00


In [11]:
# Initialize mice imputation settings
init <- mice(sdoh_data, maxit = 0) 
meth <- init$method
predM <- init$predictorMatrix
meth

# Set variables in to not be imputed
meth["person_id"] <- ""

# Columns to not use for prediction
predM[, "person_id"] <- 0  

person_id                     Black                       Mid 
                       ""                        ""                        "" 
                 Multiple                     White                       His 
                       ""                        ""                        "" 
             race_unknown                 SexGender              record_depth 
                       ""                        ""                        "" 
          visit_frequency                       age                 Education 
                       ""                        ""                        "" 
                 Own Home           Health Literacy           Housing Quality 
                    "pmm"                     "pmm"                     "pmm" 
      Housing Instability           Food Insecurity               Walkability 
                    "pmm"                     "pmm"                     "pmm" 
               Loneliness                     Crime         Physical Disorder 
                    "pmm"                     "pmm"                     "pmm" 
          Social Disorder   Everyday Discrimination    Medical Discrimination 
                    "pmm"                     "pmm"                     "pmm" 
          Social Cohesion                    Stress            Social Support 
                    "pmm"                     "pmm"                     "pmm" 
    Spiritual Experiences           Health Coverage    Healthcare Utilization 
                    "pmm"                     "pmm"                     "pmm" 
             Delayed Care         Can't afford care               Worried Pay 
                    "pmm"                     "pmm"                     "pmm" 
                  Respect Percent Poverty Threshold                      age2 
                    "pmm"                        ""                        "" 
                   AANHPI 
                       ""

In [12]:
# Perform imputation
temp_dat <- mice(sdoh_data, m = 10, seed = 123, method = meth, predictorMatrix = predM, ridge=0.001, printFlag = FALSE)

completed_dat <- suppressWarnings(suppressMessages(
  complete(temp_dat, action = "broad", include = TRUE)
))

remaining_na <- sapply(completed_dat, function(x) sum(is.na(x)))


Warning message:
“Number of logged events: 1100”


In [13]:
library(dplyr)
library(stringr)

# Define base names to take `.0` directly
use_dot0 <- c('person_id', 'Black', 'Mid', 'Multiple', 'White', 'His', 'race_unknown',
              'SexGender', 'record_depth', 'visit_frequency', 'age', 'age2', 'AANHPI')

# Get all columns ending in .0 to .10
all_pattern <- "\\.(0|1[0]?|[1-9])$"
filtered_names <- grep(all_pattern, names(completed_dat), value = TRUE)

# Get base names
base_names <- unique(str_replace(filtered_names, "\\.\\d+$", ""))

# Initialize result
averaged_dat <- data.frame(row_id = seq_len(nrow(completed_dat)))

for (var in base_names) {
  if (var %in% use_dot0) {
    colname <- paste0(var, ".0")
    averaged_dat[[var]] <- completed_dat[[colname]]
  } else {
    matching_cols <- grep(paste0("^", var, "\\.([1-9]|10)$"), names(completed_dat), value = TRUE)
    numeric_data <- completed_dat[, matching_cols]  
    averaged_dat[[var]] <- rowMeans(numeric_data, na.rm = TRUE)
  }
}

# Drop helper column if not needed
averaged_dat$row_id <- NULL



In [14]:
names(averaged_dat)

[1] "person_id"                 "Black"                    
 [3] "Mid"                       "Multiple"                 
 [5] "White"                     "His"                      
 [7] "race_unknown"              "SexGender"                
 [9] "record_depth"              "visit_frequency"          
[11] "age"                       "Education"                
[13] "Own Home"                  "Health Literacy"          
[15] "Housing Quality"           "Housing Instability"      
[17] "Food Insecurity"           "Walkability"              
[19] "Loneliness"                "Crime"                    
[21] "Physical Disorder"         "Social Disorder"          
[23] "Everyday Discrimination"   "Medical Discrimination"   
[25] "Social Cohesion"           "Stress"                   
[27] "Social Support"            "Spiritual Experiences"    
[29] "Health Coverage"           "Healthcare Utilization"   
[31] "Delayed Care"              "Can't afford care"        
[33] "Worried Pay"               "Respect"                  
[35] "Percent Poverty Threshold" "age2"                     
[37] "AANHPI"

In [15]:
# This snippet assumes that you run setup first

# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

# Replace df with THE NAME OF YOUR DATAFRAME
my_dataframe <- averaged_dat

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename <- 'imputed_individual_level_SDOH.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# store the dataframe in current workspace
write_excel_csv(my_dataframe, destination_filename)

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ./", destination_filename, " ", my_bucket, "/data/"), intern=T)

# Check if file is in the bucket
system(paste0("gsutil ls ", my_bucket, "/data/*.csv"), intern=T)


character(0)

[1] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/All_SDoH_data.csv"                                     
  [2] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/All_SDoH_data_domain_filtered_60.csv"                  
  [3] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/Case_Control_df.csv"                                   
  [4] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/Case_control_demographics_SDOH_cohort.csv"             
  [5] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/Case_control_demographics_SES_cohort.csv"              
  [6] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/Demographic_and_ancestry_covariates.csv"               
  [7] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_Afib.csv"                 
  [8] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_Asthma.csv"               
  [9] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_BreastC.csv"              
 [10] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_CHD.csv"                  
 [11] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_CKD.csv"                  
 [12] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_HyperC.csv"               
 [13] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_ProstateC.csv"            
 [14] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_t1d.csv"                  
 [15] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_t2d.csv"                  
 [16] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_Afib.csv"                
 [17] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_Asthma.csv"              
 [18] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_BreastC.csv"             
 [19] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_CHD.csv"                 
 [20] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_CKD.csv"                 
 [21] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_HyperC.csv"              
 [22] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_ProstateC.csv"           
 [23] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_t1d.csv"                 
 [24] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_t2d.csv"                 
 [25] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHW_predictions_Afib.csv"                
 [26] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHW_predictions_Asthma.csv"              
 [27] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHW_predictions_BreastC.csv"             
 [28] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHW_predictions_CHD.csv"                 
 [29] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHW_predictions_CKD.csv"                 
 [30] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHW_predictions_HyperC.csv"              
 [31] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHW_predictions_ProstateC.csv"           
 [32] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHW_predictions_t1d.csv"                 
 [33] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHW_predictions_t2d.csv"                 
 [34] "gs://fc-secure-672eeb92-4859-4ed9-9